# **4. Forecasting daily occupancy**

Chapter 3 gave us a synthetic daily history. Chapter 4 asks the practical question:
how well can we forecast the next four weeks of occupancy for a destination, and does
a smarter model beat simple rules? We build the yardstick first and the models after.

## **4.1 The forecasting task, four baselines and a ceiling**

### **4.1.1 The task**
- **Forecast:** daily occupancy for each of the five destinations, **28 days ahead**.
- **Rolling origin:** every Sunday from 2 Feb 2025 (so 13 months of history exist for
  the year-ago methods) until the last date that still has 28 days of actuals.
  At each origin a method may use only data up to and including that Sunday.
- **No leakage:** each method receives only the data up to its origin.

### **4.1.2 Four baselines (simple rules, no learning)**
1. **Last week, same weekday.** Copy the most recent matching weekday.
2. **Mean of the last 4 same weekdays.** The same idea with less noise.
3. **Last year, same weekday** (364 days earlier, so weekdays line up).
4. **Last year x recent growth.** Method 3 times the ratio of the last 28 days to the
   same 28 days a year earlier. This is the method that won in Chapter 1.

### **4.1.3 The ceiling**
An **oracle** that knows the exact generating rules but not the random noise. It is only
possible because the data are synthetic, and its error is the floor for any model: the
part of the error nobody can remove. Models will be judged on how much of the gap
between the best baseline and the oracle they close.

### **4.1.4 Metrics**
Mean absolute error in **occupancy points** (1 point = 1% of rooms), the mean error
(bias), and error by horizon band: days 1-7, 8-14 and 15-28.

### **4.1.5 Predictions (written before the run)**
1. The oracle's error is **4 to 6 points** (12% daily noise on occupancy near 0.5).
2. Every baseline lands **between 6.5 and 14 points**.
3. The **mean of the last 4 same weekdays beats last week** overall (averaging cuts noise).
4. **Last year x recent growth beats plain last year**, which should be biased low by
   about 2 to 4 points because the market grew.

### **4.1.6 Decision rule for the models in 4.3**
A model earns its place only if it (a) beats the best baseline by at least 1 point of
overall error, (b) closes at least one third of the gap between the best baseline and
the oracle, and (c) is not worse than the best baseline by more than 0.5 point in
any destination.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
import locat.forecasting.synthetic as synth

out_dir = Path("../data/processed")
DESTINATIONS = ["Maasai Mara", "Naivasha", "Diani", "Lamu", "Amboseli"]
HORIZON = 28
HISTORY_END = pd.Timestamp("2026-11-05")

# ---- data ----
syn = pd.read_csv(out_dir / "synthetic_occupancy.csv", parse_dates=["date"])
occ = syn.pivot(index="date", columns="destination", values="occupancy")[DESTINATIONS]
print("history:", occ.index.min().date(), "to", occ.index.max().date(), "| days:", len(occ))

# ---- the oracle: the generator without noise, rebuilt from the saved assumptions ----
assump = json.loads((out_dir / "synthetic_assumptions.json").read_text())
params = pd.DataFrame(assump["params"])
month_adjust = pd.DataFrame.from_dict(assump["month_adjust"], orient="index")
settings = assump["globals"]
cal = pd.read_csv(out_dir / "calendar_daily.csv", parse_dates=["date"])
seasonal = pd.read_csv(out_dir / "seasonal_index.csv")
idx_map = dict(zip(seasonal["month_number"], seasonal["seasonal_index"]))

oracle_hist = synth.generate_history(
    cal, params, month_adjust, dict(settings, noise_sd=0.0), idx_map, HISTORY_END
)
oracle = oracle_hist.pivot(index="date", columns="destination", values="occupancy")[DESTINATIONS]

# ---- rolling origins ----
FIRST_ORIGIN = pd.Timestamp("2025-02-02")  # a Sunday
last_origin = occ.index.max() - pd.Timedelta(days=HORIZON)
origins = pd.date_range(FIRST_ORIGIN, last_origin, freq="7D")
print(f"origins: {len(origins)} weekly | first {origins[0].date()} | last {origins[-1].date()}")


# ---- forecasting methods: each sees only `train` (data up to the origin) ----
def _dates(origin, horizon):
    return pd.date_range(origin + pd.Timedelta(days=1), periods=horizon)


def _lag7(horizon):
    """Smallest multiple of 7 days that reaches back to the last observed week."""
    return 7 * np.ceil(np.arange(1, horizon + 1) / 7).astype(int)


def fc_last_week(train, origin, horizon, dest):
    dates = _dates(origin, horizon)
    vals = [train.loc[d - pd.Timedelta(days=int(l))] for d, l in zip(dates, _lag7(horizon))]
    return pd.Series(vals, index=dates)


def fc_last4_weeks(train, origin, horizon, dest):
    dates = _dates(origin, horizon)
    vals = [
        np.mean([train.loc[d - pd.Timedelta(days=int(l) + 7 * j)] for j in range(4)])
        for d, l in zip(dates, _lag7(horizon))
    ]
    return pd.Series(vals, index=dates)


def fc_last_year(train, origin, horizon, dest):
    dates = _dates(origin, horizon)
    return pd.Series([train.loc[d - pd.Timedelta(days=364)] for d in dates], index=dates)


def fc_last_year_growth(train, origin, horizon, dest):
    base = fc_last_year(train, origin, horizon, dest)
    recent = train.loc[origin - pd.Timedelta(days=27): origin].sum()
    year_ago = train.loc[origin - pd.Timedelta(days=27 + 364): origin - pd.Timedelta(days=364)].sum()
    return base * (recent / year_ago)


def fc_oracle(train, origin, horizon, dest):
    return oracle.loc[_dates(origin, horizon), dest]


def evaluate(fn, name):
    parts = []
    for dest in DESTINATIONS:
        s = occ[dest]
        for o in origins:
            f = fn(s.loc[:o], o, HORIZON, dest)
            parts.append(pd.DataFrame({
                "method": name, "destination": dest, "origin": o,
                "h": np.arange(1, HORIZON + 1), "date": f.index,
                "forecast": f.values, "actual": s.loc[f.index].values,
            }))
    return pd.concat(parts, ignore_index=True)


methods = {
    "last_week_same_weekday": fc_last_week,
    "mean_last_4_same_weekdays": fc_last4_weeks,
    "last_year_same_weekday": fc_last_year,
    "last_year_x_recent_growth": fc_last_year_growth,
    "oracle_knows_the_rules": fc_oracle,
}
results = pd.concat([evaluate(fn, name) for name, fn in methods.items()], ignore_index=True)
results["error_pts"] = 100 * (results["forecast"] - results["actual"])
results["abs_error_pts"] = results["error_pts"].abs()
results["bucket"] = pd.cut(results["h"], [0, 7, 14, 28], labels=["days 1-7", "days 8-14", "days 15-28"])

# ---- results ----
order = list(methods)
mae = results.pivot_table(index="method", columns="destination", values="abs_error_pts", aggfunc="mean")[DESTINATIONS]
mae["ALL"] = results.groupby("method")["abs_error_pts"].mean()
print("\nMAE in occupancy points (1 point = 1% of rooms):")
print(mae.loc[order].round(1).to_string())

print("\nMean error (bias), points; negative = forecast too low:")
print(results.groupby("method")["error_pts"].mean().loc[order].round(2).to_string())

by_h = results.pivot_table(index="method", columns="bucket", values="abs_error_pts",
                           aggfunc="mean", observed=True).loc[order]
print("\nMAE by forecast horizon:")
print(by_h.round(1).to_string())

baselines = order[:-1]
best = mae.loc[baselines, "ALL"].idxmin()
print(f"\nBest baseline: {best} ({mae.loc[best, 'ALL']:.1f} pts) | "
      f"ceiling: {mae.loc['oracle_knows_the_rules', 'ALL']:.1f} pts")
print(f"rows evaluated per method: {len(results) // len(methods):,}")

results.drop(columns=["bucket"]).to_csv(out_dir / "backtest_baselines.csv", index=False)

history: 2024-01-01 to 2026-11-05 | days: 1040
origins: 88 weekly | first 2025-02-02 | last 2026-10-04

MAE in occupancy points (1 point = 1% of rooms):
destination                Maasai Mara  Naivasha  Diani  Lamu  Amboseli  ALL
method                                                                      
last_week_same_weekday             9.1       9.1   11.0   9.0       8.7  9.4
mean_last_4_same_weekdays          8.6       8.1   11.3   9.1       8.6  9.1
last_year_same_weekday             7.8       7.1    7.9   6.6       7.4  7.4
last_year_x_recent_growth          8.0       7.3    8.1   6.7       7.3  7.5
oracle_knows_the_rules             5.6       3.9    5.0   4.4       4.9  4.8

Mean error (bias), points; negative = forecast too low:
method
last_week_same_weekday      -0.14
mean_last_4_same_weekdays   -0.22
last_year_same_weekday      -2.61
last_year_x_recent_growth    0.10
oracle_knows_the_rules       0.12

MAE by forecast horizon:
bucket                     days 1-7  days 8-14  

## **4.2 Review of results: baselines and ceiling (step 4.1)**

### **4.2.1 Verdict**
The yardstick works. The ceiling is 4.8 points, the best baseline is 7.4 points, and
the gap between them is 2.6 points: that is the room a model has to improve. Three of
the four predictions written in 4.1.5 held. The fourth held in its cause (bias) but
failed in its effect (error), and the reason is instructive.

### **4.2.2 Prediction scorecard**

| Prediction (4.1.5) | Result | Verdict |
|---|---|---|
| Oracle error is 4 to 6 points | 4.8 overall (3.9 to 5.6 by destination) | Held |
| Every baseline is between 6.5 and 14 points | 6.6 to 11.3 by destination; 7.4 to 9.4 overall | Held (Lamu, last year, is 6.6, just inside) |
| Mean of last 4 weekdays beats last week | 9.1 against 9.4 overall | Held, but only by 0.3, and it loses in Diani (11.3 against 11.0) and Lamu (9.1 against 9.0) |
| Last year x growth beats last year; last year is biased low by 2 to 4 points | Bias -2.61, as predicted; but error 7.5 against 7.4 | Half right: bias held, error improvement failed |

### **4.2.3 What the numbers show**

#### **4.2.3.1 The ceiling**
Oracle error is 4.8 points overall and flat across horizons (4.8, 4.8, 4.8), as it
should be, since it knows the rules and nothing about the noise. It matches the
back-of-envelope figure (12% daily noise x occupancy near 0.5 x 0.8 = about 4.8).
Naivasha has the lowest floor (3.9) because its occupancy is lowest, so the same
percentage noise is fewer points; Maasai Mara has the highest (5.6).

#### **4.2.3.2 Baselines**

| Method | MAE overall | Days 1-7 | Days 8-14 | Days 15-28 |
|---|---|---|---|---|
| Last week, same weekday | 9.4 | 8.1 | 8.9 | 10.3 |
| Mean of last 4 same weekdays | 9.1 | 7.7 | 8.7 | 10.0 |
| Last year, same weekday | 7.4 | 7.4 | 7.3 | 7.4 |
| Last year x recent growth | 7.5 | 7.4 | 7.5 | 7.5 |

The two week-based rules get worse with horizon (8.1 to 10.3), because they freeze the
most recent week and miss what changes later: school breaks, long weekends, season.
The two year-ago rules are flat at about 7.4 and beat the week-based ones even in days
1-7. A likely reason is that the school calendar and the seasons fall on nearly the
same dates every year, so a year-old value carries them. That is a hypothesis. It also
means a model that uses the calendar directly has a lot to gain over last-week rules.
The week-based rules do worst in Diani (11.0 and 11.3), the destination with the
steepest year-end swing.

#### **4.2.3.3 The bias in the year-ago rule**
Last year's value is 2.61 points too low on average. That is almost exactly 5% market
growth applied to occupancy near 0.5 (0.05 x 0.5 = 2.5 points), a clean check that the
growth assumption works as designed. Scaling by recent growth removes the bias (+0.10)
but not the error (7.5 against 7.4). A rough estimate explains why: removing a 2.6-point
bias is worth about 0.3 points of error, but the growth ratio is itself noisy (it comes
from two noisy 28-day sums), and multiplying every forecast by it costs about the same.
These are approximate figures, not measured ones. It also repeats the Chapter 1 lesson
that simple growth adjustments help only when the level estimate is reliable.

#### **4.2.3.4 What a model must achieve**
The decision rule in 4.1.6 requires beating the best baseline by at least 1 point, so a
model must reach 6.4 points or better overall. That is 38% of the 2.6-point gap, so the
rule on the size of the improvement is stricter than the one-third-of-the-gap rule.

### **4.2.4 Limits**
- All results are on synthetic data; they show how methods rank when the world follows
  our rules. They do not prove how they rank on real bookings.
- The 88 origins overlap (each forecast window shares data with its neighbours), so
  the origins are not independent samples, and small differences, such as the 0.3 between
  the two week-based rules, should not be over-read.

### **4.2.5 Decision**
Move on to models. The best baseline to beat is "last year, same weekday" at 7.4.

## **4.3 Next step: a calendar-feature model, tested by removing pieces**

### **4.3.1 Why**
The baselines copy past values. A model can instead learn how each calendar driver
(weekends, school breaks, long weekends, season, growth) moves occupancy and apply
that to future dates. We start with the simplest model that could work, a ridge
regression on the logarithm of occupancy, because it is fast, stable with about two
years of data, and its results can be read.

### **4.3.2 Model**
- One model per destination, refitted every week on data up to the origin, so it
  sees exactly what a host would have seen.
- Target: log occupancy. A multiplicative world becomes additive on the log scale.
  The forecast is the exponential of the prediction, which suits an absolute-error
  measure.
- Features, all known in advance for any future date: Friday/Saturday night, school
  break, long weekend, Easter period, payday window, the log of the smoothed
  international index, three pairs of annual sine and cosine terms (a flexible
  seasonal curve), and a linear time trend (growth).
- Ridge strength is fixed at 1.0 (on standardised features) and not tuned.

### **4.3.3 Three versions (an ablation)**
1. **Full:** all features.
2. **No international index:** does the real KNBS information help?
3. **No calendar flags:** season and trend only. What do the calendar flags add?

### **4.3.4 Caveat about information leakage**
The international index was built from 2024 and 2025 arrivals, so using it for a 2025
origin lets the model see slightly ahead. It mostly encodes the annual shape, which the
sine and cosine terms can also learn, so the effect should be small. Version 2 shows how
much the index matters. For the live forecast from November 2026 there is no leakage.

### **4.3.5 Predictions (written before the run)**
1. The full model's error is **between 5.3 and 6.3 points** and passes the rule in
   4.2.3.4 (6.4 or better).
2. Removing the international index changes the error by **less than 0.3 points** in
   every destination, because the sine and cosine terms learn the same annual shape.
3. Removing the calendar flags raises the error by **at least 2 points overall**, with
   the largest rise in Naivasha (**3 points or more**), the most domestic destination
   with the strongest weekend and school effects.
4. The full model's error is **flat across horizons** (within 0.3 points), unlike the
   week-based baselines.

### **4.3.6 Decision rule**
The full model must pass the three conditions in 4.1.6. If it does not, we look at
where it fails before trying a more complex model.

In [2]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ---- calendar features, known in advance for any future date ----
feat = cal.set_index("date")
X_all = pd.DataFrame(index=feat.index)
FLAG_COLS = ["fri_sat_night", "school_out", "is_long_weekend", "easter_period", "payday_window"]
for c in FLAG_COLS:
    X_all[c] = feat[c].astype(float)
X_all["log_intl"] = np.log(synth.smooth_intl_index(pd.Series(feat.index), idx_map))
doy = feat.index.dayofyear.to_numpy()
for k in (1, 2, 3):
    X_all[f"sin{k}"] = np.sin(2 * np.pi * k * doy / 365.25)
    X_all[f"cos{k}"] = np.cos(2 * np.pi * k * doy / 365.25)
X_all["trend"] = ((X_all.index - pd.Timestamp("2024-01-01")).days / 365.25).to_numpy()
print("features:", list(X_all.columns))


def make_ridge_forecaster(alpha, columns):
    def fc(train, origin, horizon, dest):
        dates = _dates(origin, horizon)
        model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
        model.fit(X_all.loc[train.index, columns], np.log(train.values))
        pred = np.exp(model.predict(X_all.loc[dates, columns]))
        return pd.Series(np.clip(pred, 0.01, 0.98), index=dates)
    return fc


all_cols = list(X_all.columns)
variants = {
    "ridge_full": all_cols,
    "ridge_no_intl_index": [c for c in all_cols if c != "log_intl"],
    "ridge_no_calendar_flags": [c for c in all_cols if c not in FLAG_COLS],
}

new = pd.concat(
    [evaluate(make_ridge_forecaster(1.0, cols), name) for name, cols in variants.items()],
    ignore_index=True,
)
new["error_pts"] = 100 * (new["forecast"] - new["actual"])
new["abs_error_pts"] = new["error_pts"].abs()
new["bucket"] = pd.cut(new["h"], [0, 7, 14, 28], labels=["days 1-7", "days 8-14", "days 15-28"])

res = pd.concat([results, new], ignore_index=True)
order2 = baselines + list(variants) + ["oracle_knows_the_rules"]

mae2 = res.pivot_table(index="method", columns="destination", values="abs_error_pts", aggfunc="mean")[DESTINATIONS]
mae2["ALL"] = res.groupby("method")["abs_error_pts"].mean()
print("\nMAE in occupancy points:")
print(mae2.loc[order2].round(1).to_string())

print("\nMean error (bias), points:")
print(res.groupby("method")["error_pts"].mean().loc[order2].round(2).to_string())

by_h2 = res.pivot_table(index="method", columns="bucket", values="abs_error_pts",
                        aggfunc="mean", observed=True).loc[order2]
print("\nMAE by forecast horizon:")
print(by_h2.round(1).to_string())

best_name = mae2.loc[baselines, "ALL"].idxmin()
best_mae = mae2.loc[best_name, "ALL"]
oracle_mae = mae2.loc["oracle_knows_the_rules", "ALL"]
print(f"\nBest baseline: {best_name} ({best_mae:.1f}) | ceiling: {oracle_mae:.1f}")
for name in variants:
    m = mae2.loc[name, "ALL"]
    closed = (best_mae - m) / (best_mae - oracle_mae)
    worst = (mae2.loc[name, DESTINATIONS] - mae2.loc[best_name, DESTINATIONS]).max()
    print(f"{name:26s} MAE {m:.1f} | beats best baseline by {best_mae - m:+.1f} "
          f"| gap closed {closed:.0%} | worst destination vs best baseline {worst:+.1f}")

res.drop(columns=["bucket"]).to_csv(out_dir / "backtest_models.csv", index=False)

features: ['fri_sat_night', 'school_out', 'is_long_weekend', 'easter_period', 'payday_window', 'log_intl', 'sin1', 'cos1', 'sin2', 'cos2', 'sin3', 'cos3', 'trend']

MAE in occupancy points:
destination                Maasai Mara  Naivasha  Diani  Lamu  Amboseli  ALL
method                                                                      
last_week_same_weekday             9.1       9.1   11.0   9.0       8.7  9.4
mean_last_4_same_weekdays          8.6       8.1   11.3   9.1       8.6  9.1
last_year_same_weekday             7.8       7.1    7.9   6.6       7.4  7.4
last_year_x_recent_growth          8.0       7.3    8.1   6.7       7.3  7.5
ridge_full                         5.9       4.0    5.8   5.1       5.5  5.2
ridge_no_intl_index                6.0       3.9    5.7   5.2       5.6  5.3
ridge_no_calendar_flags            6.0       9.4    7.3   5.9       5.6  6.8
oracle_knows_the_rules             5.6       3.9    5.0   4.4       4.9  4.8

Mean error (bias), points:
method
last_

## **4.4 Review of results: the calendar-feature model (step 4.3)**

### **4.4.1 Verdict**
The full ridge model passes all three decision rules from 4.1.6 with room to spare. It
reaches 5.2 points of error against 7.4 for the best baseline, and the ceiling is 4.8, so
it closes 82% of the gap. It also beats the best baseline in every destination. Two of
the four predictions held outright, one held on its main point, and one missed by 0.4
points in the optimistic direction, so the ablation told us more than the headline did.

### **4.4.2 Prediction scorecard**

| Prediction (4.3.5) | Result | Verdict |
|---|---|---|
| Full model error is 5.3 to 6.3 and passes the 6.4 rule | 5.2, and it passes | Passes the rule; the range was 0.1 too pessimistic |
| Removing the international index changes error by less than 0.3 in every destination | Changes of 0.1 or less everywhere (5.2 against 5.3 overall) | Held |
| Removing the calendar flags raises error by at least 2 points overall, 3 or more in Naivasha | +1.6 overall; Naivasha +5.4 | Naivasha held strongly; the overall figure missed |
| Full model error is flat across horizons | 5.2, 5.2, 5.3 | Held |

### **4.4.3 What the numbers show**

#### **4.4.3.1 Decision rule check (full model)**
- **Beats the best baseline by at least 1 point:** yes, by 2.1 points (5.2 against 7.4).
- **Closes at least a third of the gap to the ceiling:** yes, 82%.
- **Not worse than the best baseline by more than 0.5 in any destination:** yes; it
  is better in every destination, and its weakest margin is 1.5 points.

#### **4.4.3.2 Where the remaining 0.4 points to the ceiling live**

| Destination | Ridge full | Oracle | Remaining gap |
|---|---|---|---|
| Maasai Mara | 5.9 | 5.6 | 0.3 |
| Naivasha | 4.0 | 3.9 | 0.1 |
| Diani | 5.8 | 5.0 | 0.8 |
| Lamu | 5.1 | 4.4 | 0.7 |
| Amboseli | 5.5 | 4.9 | 0.6 |

The gap is largest at the two coast destinations. A candidate explanation: in our
generator their own monthly seasons change in steps at month boundaries (for example
Diani goes from 1.20 in December to 1.15 in January, and drops to 0.75 in May), and
three sine and cosine pairs cannot reproduce steps. That is a hypothesis. It is also a
generator artifact, since real seasons change more smoothly. Chasing it would mean
fitting our own simplification, so we do not.

#### **4.4.3.3 The ablation: what each piece contributes**

| Destination | Full | No international index | No calendar flags | Flags effect | Domestic share |
|---|---|---|---|---|---|
| Maasai Mara | 5.9 | 6.0 | 6.0 | +0.1 | 40% |
| Naivasha | 4.0 | 3.9 | 9.4 | +5.4 | 90% |
| Diani | 5.8 | 5.7 | 7.3 | +1.5 | 55% |
| Lamu | 5.1 | 5.2 | 5.9 | +0.8 | 55% |
| Amboseli | 5.5 | 5.6 | 5.6 | +0.1 | 35% |

- **The calendar flags matter in proportion to the domestic share of guests,** which is
  what the generator was built to do. In a destination that mostly hosts Kenyans
  (Naivasha) the flags cut the error from 9.4 to 4.0; in the safari destinations they
  hardly matter, and season carries the forecast. I mis-estimated the overall figure
  because I treated every destination like Naivasha.
- **The international index adds nothing beyond the seasonal curve the model learns from
  the destination's own history** (changes of 0.1 or less). That is a statement about this
  data. The index will still matter for a **new host with no booking history**, for whom
  it is the only seasonal information, so we keep it in the model for them.

#### **4.4.3.4 Bias**
The full model is 0.41 points low on average. That is close to what taking the
exponential of a log-scale prediction produces (it gives the median, which lies below
the mean by about 0.7% of occupancy, roughly 0.4 points at occupancy near 0.5).
This is the right choice for absolute error. If the price advisor later needs the
expected value, we add a small correction.

### **4.4.4 Limits**
- The generator is close to the model's own form (multiplicative effects, a smooth
  seasonal curve), so part of this success is by construction. The real test is real
  bookings after launch.
- The international-index leakage caveat from 4.3.4 stands, and the ablation shows it
  did not matter here.
- The 88 origins overlap, so the figures are not 88 independent trials.

### **4.4.5 Decision**
Adopt the ridge model as the forecaster. We drop gradient boosting for now: the model
is within 0.4 points of the best possible score, so there is little left for a more
complex model to gain on this data, and it could not be told apart from noise. We revisit
it when real bookings exist. Next: check that the model's *explanations* match the
answer key (4.5), then add prediction intervals, then move the model into the project.

## **4.5 Next step: do the model's effects match the answer key?**

### **4.5.1 Why**
The price advisor will explain itself, for example "school holidays lift demand at your
place by about 30%". Those sentences are only honest if the model recovered the true
effects. Predicting well is not the same as explaining well: effects can trade off
against each other, such as school breaks against the annual season, and still forecast
well.

### **4.5.2 Method**
1. Fit the full model for each destination on all history (to 5 Nov 2026).
2. Read each flag's effect as a multiplier: the exponential of its coefficient in the
   original units (the scaling is undone).
3. Compare with the answer key from 3.10 (noise-free, fixed-effects). An effect counts as
   **recovered** if it lies within 2 seed-to-seed standard deviations of the key.
4. Easter and payday have no key, so we compare with the approximate mixing-rule target:
   international share + (1 - international share) x the assumed boost.
5. Check the implied yearly growth against the assumed 5%.

### **4.5.3 Predictions (written before the run)**
1. **Weekend:** all 5 destinations recovered (the effect is strong and measured often).
2. **School break:** at most 3 of 5 recovered. School breaks fall on nearly the same dates
   each year, so they are partly confounded with the annual seasonal curve.
3. **Long weekend:** at least 3 of 5 recovered (the tolerance is wide, 0.07 to 0.08).
4. **Payday:** within 0.03 of the approximate target in every destination. **Easter:**
   within 0.08 in at least 3 of 5 (only about 15 training days, and it overlaps long
   weekends and the April school break).
5. **Implied growth:** between 3% and 8% in every destination.

### **4.5.4 Decision rule**
- If at least 12 of the 15 key cells are recovered, the advisor may quote the model's
  effects directly.
- Otherwise it quotes only the effects that were recovered, and for the others we say
  "around X%" using the answer key's range, or leave them out of the explanation.

In [3]:
# ---- fit the full model on all history; read each flag's effect as a multiplier ----
ref = pd.read_csv(out_dir / "synthetic_uplift_reference.csv", index_col=[0, 1])
key = ref.loc["noise_free_fe"].loc[DESTINATIONS]
key_sd = ref.loc["sd_20_seeds_fe"].loc[DESTINATIONS]

rows, growth_implied = {}, {}
for dest in DESTINATIONS:
    model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    model.fit(X_all.loc[occ.index, all_cols], np.log(occ[dest].values))
    coef = pd.Series(
        model.named_steps["ridge"].coef_ / model.named_steps["standardscaler"].scale_,
        index=all_cols,
    )
    rows[dest] = np.exp(coef[FLAG_COLS])
    growth_implied[dest] = np.exp(coef["trend"]) - 1
effects = pd.DataFrame(rows).T.loc[DESTINATIONS]

model_vs_key = pd.DataFrame({
    "fri_sat_vs_other": effects["fri_sat_night"],
    "school_out_vs_in": effects["school_out"],
    "long_wknd_vs_normal_fri_sat": effects["is_long_weekend"],
})
diff = model_vs_key - key
z = diff / key_sd
within = z.abs() <= 2

print("Model effects (multipliers):")
print(model_vs_key.round(2).to_string())
print("\nAnswer key (noise-free):")
print(key.round(2).to_string())
print("\nDifference (model - key):")
print(diff.round(2).to_string())
print("\nDifference in seed-to-seed standard deviations:")
print(z.round(1).to_string())
print(f"\nCells within 2 sd of the key: {int(within.sum().sum())} of {within.size}")
print(within.sum().to_string())

# ---- Easter and payday: no key, so compare with the approximate mixing-rule target ----
s = params.loc["intl_share"]
approx = pd.DataFrame({
    "easter_model": effects["easter_period"],
    "easter_target": s + (1 - s) * settings["easter_extra"],
    "payday_model": effects["payday_window"],
    "payday_target": s + (1 - s) * settings["payday_boost"],
}).loc[DESTINATIONS]
print("\nEaster and payday against the approximate target:")
print(approx.round(3).to_string())

print("\nImplied yearly growth (assumed 5%):")
print(pd.Series(growth_implied).loc[DESTINATIONS].round(3).to_string())

Model effects (multipliers):
             fri_sat_vs_other  school_out_vs_in  long_wknd_vs_normal_fri_sat
Maasai Mara              1.04              1.05                         1.09
Naivasha                 1.52              1.26                         1.41
Diani                    1.12              1.25                         1.19
Lamu                     1.10              1.14                         1.18
Amboseli                 1.04              1.04                         1.07

Answer key (noise-free):
             fri_sat_vs_other  school_out_vs_in  long_wknd_vs_normal_fri_sat
destination                                                                 
Maasai Mara              1.05              1.11                         1.08
Naivasha                 1.57              1.32                         1.45
Diani                    1.15              1.28                         1.22
Lamu                     1.12              1.21                         1.16
Amboseli             

## **4.6 Review of results: do the model's effects match the answer key? (step 4.5)**

### **4.6.1 Verdict**
By the decision rule as written, the test failed: 9 of 15 key cells were recovered and
the rule needs 12. I do not reinterpret the rule to make it pass. But the pattern in the
numbers shows that the yardstick itself is partly responsible for the shortfall, so this
result cannot say how much of it belongs to the model. Step 4.7 repeats the test against
a cleaner yardstick that I set up in advance. Until then the price advisor gives
qualitative reasons ("school holidays", "long weekend"), not percentages.

### **4.6.2 Prediction scorecard**

| Prediction (4.5.3) | Result | Verdict |
|---|---|---|
| Weekend: all 5 recovered | 3 of 5 (Naivasha -3.5 sd, Diani -3.1 sd are outside) | Missed |
| School break: at most 3 of 5 recovered | 1 of 5 (only Diani) | Held |
| Long weekend: at least 3 of 5 recovered | 5 of 5, all within +/-0.04 | Held, better than predicted |
| Payday: within 0.03 of the target everywhere | Within 0.02 everywhere | Held |
| Easter: within 0.08 in at least 3 of 5 | 4 of 5 (Diani is -0.086) | Held |
| Implied growth between 3% and 8% | 4.2% to 6.5% | Held |

### **4.6.3 What the numbers show**

#### **4.6.3.1 Where the 9 of 15 come from**
Recovered cells: weekend 3, school break 1, long weekend 5. In 13 of the 15 cells the
model's value is at or below the key, which is a systematic pattern, not scatter.

#### **4.6.3.2 Weekend effect: the key may be the problem**

| Destination | Model | Mixing-rule target | Key |
|---|---|---|---|
| Maasai Mara | 1.04 | 1.04 | 1.05 |
| Naivasha | 1.52 | 1.54 | 1.57 |
| Diani | 1.12 | 1.14 | 1.15 |
| Lamu | 1.10 | 1.11 | 1.12 |
| Amboseli | 1.04 | 1.04 | 1.04 |

The model matches the mixing-rule target to within 0.02 everywhere, while the key sits
0.01 to 0.05 above the target. The two "failures" (Naivasha -0.05, Diani -0.04) look
large in standard deviations only because the seed-to-seed spread is tiny (0.011 to
0.014). A likely cause, still unverified, is that the key came from a single-flag
estimate, which absorbs the long-weekend boost carried by Friday and Saturday nights. A
joint model separates the two, so it reads lower. I also made a mistake in the test
design: in 3.11.3 I said the key is good to a few hundredths, then set the tolerance at
two seed-spreads (0.02 to 0.03), tighter than the yardstick itself.

#### **4.6.3.3 School-break effect: a real shortfall in three destinations**

| Destination | Model | Mixing-rule target | Key |
|---|---|---|---|
| Maasai Mara | 1.05 | 1.10 | 1.11 |
| Naivasha | 1.26 | 1.27 | 1.32 |
| Diani | 1.25 | 1.28 | 1.28 |
| Lamu | 1.14 | 1.19 | 1.21 |
| Amboseli | 1.04 | 1.09 | 1.10 |

The model is below the target by about 0.05 for Maasai Mara, Lamu and Amboseli, and
within 0.03 for Naivasha and Diani. School breaks fall on nearly the same dates every
year, so they overlap with the seasonal curve, and in destinations with many
international guests the school effect is a small signal (5 to 10%) next to large
seasonal swings. That would explain why it is hardest to recover there. It remains a
hypothesis.

#### **4.6.3.4 Long weekend**
All five cells are within 0.04 of the key (differences +0.01, -0.04, -0.03, +0.01,
-0.02). This effect is recovered everywhere.

#### **4.6.3.5 Easter and payday**

| Destination | Easter model / target | Payday model / target |
|---|---|---|
| Maasai Mara | 1.029 / 1.040 | 1.011 / 1.020 |
| Naivasha | 1.135 / 1.090 | 1.036 / 1.045 |
| Diani | 0.969 / 1.055 | 1.034 / 1.028 |
| Lamu | 1.110 / 1.055 | 1.008 / 1.028 |
| Amboseli | 1.025 / 1.035 | 1.006 / 1.018 |

Payday is recovered well (within 0.02). Easter is noisy: Diani's value is below 1.0,
which cannot be true in our generator, where the Easter boost is 1.10 or more, and
Lamu is 0.055 too high. Only three Easters (about 15 days) are in the training data and
they overlap long weekends and the April school break. We use Easter as a control
variable only and never quote it.

#### **4.6.3.6 Growth**
Implied growth is 5.2%, 4.2%, 4.7%, 4.9% and 6.5% against the assumed 5%, so the
trend term works. Amboseli is the highest.

### **4.6.4 Limits**
- The key is a single-flag estimate, so it is a yardstick good to a few hundredths, not
  exact truth, and the tolerance used here ignored that.
- The model's own estimation noise has not been measured (only one dataset was fitted).

### **4.6.5 Decision**
Do not claim that the model's explanations are verified. Run a cleaner test (4.7) with
two changes: an exact yardstick computed from the generator's formula, and a 20-seed
spread of the model's own estimates, so bias and noise can be separated.

## **4.7 Next step: exact structural truth and the model's own noise**

### **4.7.1 Why**
The test in 4.5 compared the model with an estimate (the key), and the estimate has its
own error. The generator's formula lets us compute the exact effect of each flag, with
no estimator involved, and a 20-seed run measures how much the model's estimates move
from noise alone.

### **4.7.2 Method**
1. **Exact structural effect.** For each destination and flag, take every history day
   on which the flag is on and compute the ratio of occupancy with the flag to
   occupancy with that one boost switched off, holding everything else fixed (including
   the normalisation). Growth, season and noise cancel in the ratio. Average the
   ratios geometrically, which matches a model on the log scale.
2. **The model's estimates over 20 seeds** (seeds 100 to 119): refit on each synthetic
   history and read the same effects.
3. **Compare:** bias = 20-seed mean minus the exact effect; noise = the seed-to-seed
   standard deviation.
4. Also compute how far the old key was from the exact effect.

### **4.7.3 Predictions (written before the run)**
1. The old key sits **above** the exact effect in most cells, by up to about 0.05, with the
   largest gaps in Naivasha.
2. **Weekend:** the model is within 0.03 of the exact effect in all 5 destinations.
3. **School break:** the model is **below** the exact effect by 0.03 to 0.06 in Maasai
   Mara, Lamu and Amboseli (a bias, not noise), and within 0.03 in Naivasha and Diani.
4. **Long weekend:** within 0.04 of the exact effect in all 5.
5. Seed-to-seed spread of the model's estimates is at most 0.03 for weekend, 0.04 for
   school break and 0.06 for long weekend.

### **4.7.4 Decision rule: how each effect may be quoted by the advisor**
- **Quote as a number** if the absolute bias is 0.03 or less and the spread is 0.03 or
  less.
- **Quote as a range** if the bias and the spread are both 0.06 or less.
- **Qualitative only** otherwise.

In [4]:
def structural_effects(dest):
    """Exact effect of each flag on the days it is on, from the generator's formula.
    With flag: s*I + (1-s)*D/m. Without: the same with that boost removed from D.
    Growth, own season and noise multiply both and cancel in the ratio."""
    p = params[dest]
    c = cal[cal["date"] <= HISTORY_END].reset_index(drop=True)
    I = synth.smooth_intl_index(c["date"], idx_map)
    boosts = {
        "fri_sat_night": p["weekend_boost"],
        "school_out": p["school_boost"],
        "is_long_weekend": p["longwknd_boost"],
        "easter_period": settings["easter_extra"],
        "payday_window": settings["payday_boost"],
    }
    factors = {f: np.where(c[f] == 1, b, 1.0) for f, b in boosts.items()}
    D = np.prod(list(factors.values()), axis=0)
    m = D.mean()
    s = p["intl_share"]
    with_flag = s * I + (1 - s) * D / m
    out = {}
    for f in boosts:
        on = (c[f] == 1).to_numpy()
        without = s * I + (1 - s) * (D / factors[f]) / m
        out[f] = np.exp(np.mean(np.log(with_flag[on] / without[on])))
    return pd.Series(out).reindex(FLAG_COLS)


def fit_effects(occ_wide):
    rows, growth = {}, {}
    for dest in DESTINATIONS:
        model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
        model.fit(X_all.loc[occ_wide.index, all_cols], np.log(occ_wide[dest].values))
        coef = pd.Series(
            model.named_steps["ridge"].coef_ / model.named_steps["standardscaler"].scale_,
            index=all_cols,
        )
        rows[dest] = np.exp(coef[FLAG_COLS])
        growth[dest] = np.exp(coef["trend"]) - 1
    return pd.DataFrame(rows).T.loc[DESTINATIONS], pd.Series(growth).loc[DESTINATIONS]


truth = pd.DataFrame({d: structural_effects(d) for d in DESTINATIONS}).T.loc[DESTINATIONS]
est42, _ = fit_effects(occ)

seeds = list(range(100, 120))
runs = []
for seed in seeds:
    hist = synth.generate_history(
        cal, params, month_adjust, dict(settings, seed=seed), idx_map, HISTORY_END
    )
    wide = hist.pivot(index="date", columns="destination", values="occupancy")[DESTINATIONS]
    runs.append(fit_effects(wide)[0])

all_e = pd.concat(runs, keys=seeds, names=["seed", "destination"])
mean_e = all_e.groupby("destination").mean().loc[DESTINATIONS]
sd_e = all_e.groupby("destination").std().loc[DESTINATIONS]
bias = mean_e - truth

print("Exact structural effect (the new yardstick):")
print(truth.round(3).to_string())

key_cols = {"fri_sat_night": "fri_sat_vs_other",
            "school_out": "school_out_vs_in",
            "is_long_weekend": "long_wknd_vs_normal_fri_sat"}
old_vs_truth = pd.DataFrame({f: key[k] - truth[f] for f, k in key_cols.items()})
print("\nOld key minus exact effect:")
print(old_vs_truth.round(3).to_string())

print("\nModel (seed 42) minus exact effect:")
print((est42 - truth).round(3).to_string())
print("\nBias: 20-seed model average minus exact effect:")
print(bias.round(3).to_string())
print("\nNoise: seed-to-seed spread of the model's estimates:")
print(sd_e.round(3).to_string())


def quote_label(b, s):
    if abs(b) <= 0.03 and s <= 0.03:
        return "number"
    if abs(b) <= 0.06 and s <= 0.06:
        return "range"
    return "qualitative"


labels = pd.DataFrame(
    {f: [quote_label(bias.loc[d, f], sd_e.loc[d, f]) for d in DESTINATIONS] for f in FLAG_COLS},
    index=DESTINATIONS,
)
print("\nHow the advisor may quote each effect:")
print(labels.to_string())

Exact structural effect (the new yardstick):
             fri_sat_night  school_out  is_long_weekend  easter_period  payday_window
Maasai Mara          1.040       1.095            1.065          1.050          1.020
Naivasha             1.531       1.266            1.410          1.094          1.045
Diani                1.133       1.255            1.205          1.068          1.027
Lamu                 1.108       1.183            1.147          1.066          1.027
Amboseli             1.035       1.083            1.077          1.045          1.017

Old key minus exact effect:
             fri_sat_night  school_out  is_long_weekend
destination                                            
Maasai Mara          0.008       0.019            0.011
Naivasha             0.042       0.053            0.035
Diani                0.022       0.028            0.018
Lamu                 0.016       0.024            0.018
Amboseli             0.008       0.019            0.010

Model (seed 42) m

## **4.8 Review of results: exact structural truth and the model's own noise (step 4.7)**

### **4.8.1 Verdict**
The cleaner yardstick settles the question left open in 4.6. Most of the earlier
shortfall belonged to the old answer key, which turns out to be biased high in all 15
cells. Against the exact effects, the model recovers the weekend and long-weekend effects
well, shows a small low bias on the school-break effect, and cannot pin down the Easter
effect at all. All five predictions in 4.7.3 held on the numbers stated, with one
qualification on school breaks (see 4.8.3.3). Using the quoting rule fixed in 4.7.4, 13 of
the 15 main effects may be quoted as numbers and the other 2 as ranges.

### **4.8.2 Prediction scorecard**

| Prediction (4.7.3) | Result | Verdict |
|---|---|---|
| The old key is above the exact effect, by up to about 0.05, most in Naivasha | Above in all 15 cells, by 0.008 to 0.053; Naivasha largest | Held |
| Weekend: model within 0.03 of the exact effect everywhere | Within 0.014 everywhere | Held |
| School break: model low by 0.03 to 0.06 in Maasai Mara, Lamu, Amboseli; within 0.03 in Naivasha, Diani | Seed 42: -0.041, -0.040, -0.039 and -0.001, -0.010 | Held on the numbers; the 20-seed bias points to a different set (4.8.3.3) |
| Long weekend: within 0.04 everywhere | Within 0.032 everywhere | Held |
| Seed-to-seed spread: at most 0.03 (weekend), 0.04 (school), 0.06 (long weekend) | 0.009 to 0.014; 0.010 to 0.022; 0.020 to 0.029 | Held |

### **4.8.3 What the numbers show**

#### **4.8.3.1 The old key was biased high**
The old key minus the exact effect is positive in every one of the 15 cells (0.008 to
0.053), with the largest values in Naivasha (0.042, 0.053, 0.035). That matches the
earlier suspicion that a single-flag estimate absorbs boosts that travel with the flag.
The test shows the key was too high, not why, so the cause is still a hypothesis.

#### **4.8.3.2 Weekend effect: clean**
The seed-42 model is within 0.014 of the exact effect in all destinations. Across 20
seeds the bias is 0.005 or less and the spread 0.009 to 0.014. The model measures this
effect accurately and precisely.

#### **4.8.3.3 School-break effect: small low bias**

| Destination | Exact effect | Seed 42 minus exact | 20-seed bias | Spread |
|---|---|---|---|---|
| Maasai Mara | 1.095 | -0.041 | -0.017 | 0.019 |
| Naivasha | 1.266 | -0.001 | -0.010 | 0.022 |
| Diani | 1.255 | -0.010 | -0.036 | 0.020 |
| Lamu | 1.183 | -0.040 | -0.037 | 0.010 |
| Amboseli | 1.083 | -0.039 | -0.030 | 0.015 |

All five biases are negative, from -0.010 to -0.037. My prediction named the wrong set of
destinations. Seed 42 flattered Diani (-0.010 against a true bias of -0.036) and
exaggerated Maasai Mara (-0.041 against -0.017). The 20-seed averages place the bias
in Diani, Lamu and Amboseli (-0.030 to -0.037) and show it is small in Maasai Mara and
Naivasha. The seed-42 data is the single dataset we actually use, so its school-break
estimates for Maasai Mara, Lamu and Amboseli read about 0.04 low. A candidate cause is
overlap with the smooth seasonal curve plus ridge shrinkage, since school breaks fall on
nearly the same dates every year. It is untested, and the effect is small.

#### **4.8.3.4 Long weekend: recovered**
The 20-seed bias is -0.024 to +0.017, the spread 0.020 to 0.029, and the seed-42
differences are within 0.032. Recovered everywhere.

#### **4.8.3.5 Easter and payday**
Easter's spread is 0.047 to 0.061 and its bias runs from -0.020 to +0.046, so a single
Easter estimate is nearly meaningless: it is a control variable and not a result.
Payday is estimated cleanly (bias 0.003 or less, spread 0.012 to 0.013), but the effect
itself is small (1.7% to 4.5%).

#### **4.8.3.6 How the advisor may quote each effect**

| Destination | Weekend | School break | Long weekend | Easter | Payday |
|---|---|---|---|---|---|
| Maasai Mara | number | number | number | range | number |
| Naivasha | number | number | number | range | number |
| Diani | number | range | number | qualitative | number |
| Lamu | number | range | number | range | number |
| Amboseli | number | number | number | range | number |

### **4.8.4 Limits**
- The exact effect is exact only for our generator. With real bookings there is no such
  yardstick, so the quoting labels will have to be earned again on real data.
- A "number" label means a bias of 0.03 or less and a spread of 0.03 or less. It does not
  mean the effect is exact. The advisor should round to the nearest 5%.

### **4.8.5 Decision**
Accept the model's effects for the advisor's explanations, under the labels above.
No further model tuning is needed for this purpose. Next, a forecast must carry its
uncertainty: the advisor should know how sure it is.

## **4.9 Next step: prediction intervals, checked by backtest**

### **4.9.1 Why**
A single number such as "62% full" invites false confidence. A host deciding on a price
needs to know whether that is 55 to 70% or 40 to 85%. Intervals also let the advisor stay
quiet when the forecast is too uncertain to act on.

### **4.9.2 Method**
1. For each destination, collect the model's past errors on the log scale, defined as
   log(actual / forecast), but **only errors whose target date has already passed at the
   origin**, so no future information is used.
2. At each origin take the 10th and 90th percentiles of those errors. The 80% interval is
   the forecast multiplied by their exponentials, capped at 98%.
3. Start evaluating once a destination has at least 150 past errors.
4. Measure **coverage** (the share of actuals inside the interval; the target is 80%) and
   **width** in occupancy points, by destination, horizon band, period and day type.
5. Save the final percentiles from all backtest errors for the app to use.

### **4.9.3 Predictions (written before the run)**
1. Overall coverage is **between 77% and 86%**.
2. Coverage differs by **no more than 5 points** between horizon bands (the error is flat
   across horizon).
3. Average width is **14 to 20 points** (12% noise around occupancy near 0.5).
4. Coverage is **higher in the second half** of the evaluated origins than in the first,
   because early errors, made with only 13 months of training data, inflate the early
   intervals... and so we expect over-coverage later.
5. Coverage on long-weekend days is **below** coverage on other days (peaks sit near the
   98% cap and the noise is larger in absolute terms).

### **4.9.4 Decision rule**
- **Accept** the intervals if overall coverage is between 76% and 84%, no destination is
  below 72% or above 88%, and the horizon bands lie within 5 points of each other.
- Otherwise recalibrate by scaling the percentiles. We do not blame the model first.

In [5]:
rf = res[res["method"] == "ridge_full"].copy()
rf["log_ratio"] = np.log(rf["actual"] / rf["forecast"])

Q_LO, Q_HI, MIN_N = 0.10, 0.90, 150

parts = []
for dest in DESTINATIONS:
    d = rf[rf["destination"] == dest]
    for o in origins:
        known = d.loc[d["date"] <= o, "log_ratio"]  # errors already observed at this origin
        if len(known) < MIN_N:
            continue
        lo_q, hi_q = known.quantile([Q_LO, Q_HI])
        cur = d[d["origin"] == o].copy()
        cur["lower"] = np.clip(cur["forecast"] * np.exp(lo_q), 0.0, 0.98)
        cur["upper"] = np.clip(cur["forecast"] * np.exp(hi_q), 0.0, 0.98)
        parts.append(cur)

ivl = pd.concat(parts, ignore_index=True)
ivl["covered"] = (ivl["actual"] >= ivl["lower"]) & (ivl["actual"] <= ivl["upper"])
ivl["width_pts"] = 100 * (ivl["upper"] - ivl["lower"])
ivl["band"] = pd.cut(ivl["h"], [0, 7, 14, 28], labels=["days 1-7", "days 8-14", "days 15-28"])
ivl = ivl.merge(cal[["date", "fri_sat_night", "is_long_weekend", "school_out"]], on="date", how="left")

first_origin = ivl["origin"].min()
mid = ivl["origin"].sort_values().iloc[len(ivl) // 2]
ivl["half"] = np.where(ivl["origin"] < mid, "first half of origins", "second half of origins")

print(f"intervals evaluated from origin {first_origin.date()} | rows: {len(ivl):,}")
print(f"overall coverage (target 80%): {ivl['covered'].mean():.1%} | mean width: {ivl['width_pts'].mean():.1f} points")

print("\nBy destination:")
print(ivl.groupby("destination")[["covered", "width_pts"]].mean().loc[DESTINATIONS]
      .assign(covered=lambda t: (100 * t["covered"]).round(1), width_pts=lambda t: t["width_pts"].round(1))
      .to_string())

print("\nBy horizon band (coverage %):")
print((100 * ivl.groupby("band", observed=True)["covered"].mean()).round(1).to_string())

print("\nBy period (coverage %):")
print((100 * ivl.groupby("half")["covered"].mean()).round(1).to_string())

print("\nBy day type (coverage %):")
for col, label in [("fri_sat_night", "Fri/Sat night"), ("is_long_weekend", "long weekend"), ("school_out", "school break")]:
    g = 100 * ivl.groupby(col)["covered"].mean()
    print(f"  {label}: on {g.get(1, float('nan')):.1f} | off {g.get(0, float('nan')):.1f}")

# Final percentiles from all backtest errors, saved for the app
final_q = rf.groupby("destination")["log_ratio"].quantile([Q_LO, Q_HI]).unstack().loc[DESTINATIONS]
final_q.columns = ["q10", "q90"]
final_q.to_csv(out_dir / "forecast_interval_quantiles.csv")
print("\nFinal log-error percentiles (saved):")
print(final_q.round(3).to_string())

intervals evaluated from origin 2025-03-23 | rows: 11,340
overall coverage (target 80%): 80.7% | mean width: 17.5 points

By destination:
             covered  width_pts
destination                    
Maasai Mara     81.3       20.5
Naivasha        80.6       12.5
Diani           79.1       18.3
Lamu            81.0       17.0
Amboseli        81.5       19.4

By horizon band (coverage %):
band
days 1-7      80.5
days 8-14     80.8
days 15-28    80.8

By period (coverage %):
half
first half of origins     79.8
second half of origins    81.7

By day type (coverage %):
  Fri/Sat night: on 79.1 | off 81.4
  long weekend: on 80.9 | off 80.7
  school break: on 81.9 | off 80.2

Final log-error percentiles (saved):
               q10    q90
destination              
Maasai Mara -0.168  0.172
Naivasha    -0.129  0.157
Diani       -0.197  0.137
Lamu        -0.156  0.185
Amboseli    -0.177  0.167
